In [1]:
!pip uninstall -y torch torchvision torchaudio transformers peft trl accelerate bitsandbytes autoawq numpy scipy -q
!pip install --no-cache-dir --ignore-installed "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" "tokenizers==0.20.3" "huggingface_hub==0.27.1" "datasets==3.1.0" -q
!pip install --no-cache-dir --ignore-installed "autoawq==0.2.7.post2" -q
!pip install --no-cache-dir --ignore-installed "numpy==2.0.2" "scipy==1.13.1" -q
!pip install --no-cache-dir --ignore-installed --index-url https://download.pytorch.org/whl/cu124 "torch==2.5.1" -q
!pip install --no-cache-dir --force-reinstall --no-deps "transformers==4.46.3" -q
print("Now: Run → Restart & clear outputs, then run cells 2-5")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 261.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 296.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 326.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 307.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 237.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 285.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 330.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 276.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 352.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 209.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install --no-cache-dir --force-reinstall --no-deps "huggingface_hub==0.27.1" "tokenizers==0.20.3" "transformers==4.46.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 141.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.3
    Uninstalling transformers-4.46.3:
      Successfully uninstalled transformers-4.46.3
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [3]:
import os, json, gc, re, time, glob
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
AWQ_MODEL  = "ayushgupta7777/sentinelops-mistral7b-awq"
N_SAMPLES, MAX_NEW_TOKENS = 10, 768
OUT_DIR = Path("/kaggle/working"); OUT_DIR.mkdir(exist_ok=True)

hits = glob.glob("/kaggle/input/**/sft_val.jsonl", recursive=True)
assert hits, "sft_val.jsonl not found — attach val dataset to notebook"
VAL_PATH = hits[0]; print("val:", VAL_PATH)

val: /kaggle/input/datasets/ayushgupta07xx/sentinelops-sft/sft_val.jsonl


In [4]:
samples = []
with open(VAL_PATH) as f:
    for i, line in enumerate(f):
        if i >= N_SAMPLES: break
        samples.append(json.loads(line))
print(f"Loaded {len(samples)} samples")

def build_prompt(s):
    return f"<s>[INST] {s['instruction']}\n\n{s['input']} [/INST]"

Loaded 10 samples


In [5]:
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

2.5.1+cu124 True


In [6]:
from awq import AutoAWQForCausalLM

def generate_base(model_id, label):
    print(f"\n=== {label}: {model_id} ===")
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map={"": 0}, torch_dtype=torch.float16)
    model.eval()
    return _run(model, tok, label)

def generate_awq(model_id, label):
    print(f"\n=== {label}: {model_id} ===")
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoAWQForCausalLM.from_quantized(model_id, device_map={"": 0}, fuse_layers=False, safetensors=True)
    return _run(model.model, tok, label)

def _run(model, tok, label):
    outs = []
    for i, s in enumerate(samples):
        torch.manual_seed(42 + i)
        prompt = build_prompt(s)
        inp = tok(prompt, return_tensors="pt", truncation=True, max_length=3072).to("cuda:0")
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=True, temperature=0.7, top_p=0.9, top_k=50,
                                 repetition_penalty=1.1,
                                 pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  [{i+1}/{N_SAMPLES}] {time.time()-t0:.1f}s  {len(gen.split())}w")
        outs.append(gen)
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return outs

base_outputs = generate_base(BASE_MODEL, "BASE")
awq_outputs  = generate_awq(AWQ_MODEL,  "AWQ")

2026-04-20 01:39:57.476365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776649197.692484      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776649197.754506      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776649198.263412      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776649198.263449      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776649198.263452      23 computation_placer.cc:177] computation placer alr


=== BASE: mistralai/Mistral-7B-Instruct-v0.3 ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

Replacing layers...: 100%|██████████| 32/32 [00:10<00:00,  2.95it/s]


  [1/10] 78.8s  301w
  [2/10] 71.0s  569w
  [3/10] 70.5s  500w
  [4/10] 70.8s  544w
  [5/10] 72.8s  315w
  [6/10] 70.7s  532w
  [7/10] 71.2s  595w
  [8/10] 70.4s  566w
  [9/10] 70.3s  469w
  [10/10] 73.9s  333w


In [7]:
SECTIONS = {
    "root_cause":  r"root\s*cause",
    "impact":      r"\bimpact\b|customer\s*impact|service\s*impact",
    "remediation": r"remediation|mitigation|resolution|action\s*(items|taken)",
    "learnings":   r"learnings|lessons\s*learned|takeaways|follow[- ]?ups?",
}
def score(text):
    tl = text.lower()
    sec = {k: int(bool(re.search(v, tl))) for k, v in SECTIONS.items()}
    return {"sections": sec, "section_score": sum(sec.values()),
            "word_count": len(text.split())}
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return round(len(sa & sb) / max(len(sa | sb), 1), 3)

scores, md = [], ["# A/B Sanity Check — Base Mistral vs SentinelOps AWQ\n"]
for i, s in enumerate(samples):
    bs, aw = score(base_outputs[i]), score(awq_outputs[i])
    bs["ref_jaccard"] = jaccard(base_outputs[i], s["output"])
    aw["ref_jaccard"] = jaccard(awq_outputs[i], s["output"])
    scores.append({"idx": i, "source_id": s.get("source_id"), "base": bs, "awq": aw})
    md += [
        f"\n---\n## Sample {i+1}: `{s.get('source_id','?')}`\n",
        f"**Input excerpt:** {s['input'][:300]}...\n",
        f"\n**Base Mistral** (sections={bs['section_score']}/4, words={bs['word_count']}, ref_jac={bs['ref_jaccard']})\n",
        f"```\n{base_outputs[i]}\n```\n",
        f"\n**SentinelOps AWQ** (sections={aw['section_score']}/4, words={aw['word_count']}, ref_jac={aw['ref_jaccard']})\n",
        f"```\n{awq_outputs[i]}\n```\n",
    ]

n = len(scores)
summary = f"""
## Aggregate (n={n})
| Metric | Base Mistral | SentinelOps AWQ |
|---|---|---|
| Section coverage (sum /{4*n}) | {sum(x['base']['section_score'] for x in scores)} | {sum(x['awq']['section_score'] for x in scores)} |
| Avg ref Jaccard | {sum(x['base']['ref_jaccard'] for x in scores)/n:.3f} | {sum(x['awq']['ref_jaccard'] for x in scores)/n:.3f} |
| Avg word count | {sum(x['base']['word_count'] for x in scores)/n:.0f} | {sum(x['awq']['word_count'] for x in scores)/n:.0f} |
"""
md.insert(1, summary)
(OUT_DIR/"ab_results.md").write_text("\n".join(md))
(OUT_DIR/"ab_scores.json").write_text(json.dumps(scores, indent=2))
print(summary)


## Aggregate (n=10)
| Metric | Base Mistral | SentinelOps AWQ |
|---|---|---|
| Section coverage (sum /40) | 36 | 16 |
| Avg ref Jaccard | 0.123 | 0.238 |
| Avg word count | 403 | 472 |

